In [1]:
# 1. Importar librerías de datos y Machine Learning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 2. Reconstruir rápidamente la Tabla Maestra (Data Prep)
df_est = pd.read_csv('../data/01_raw/estudiantes.csv')
df_insc = pd.read_csv('../data/01_raw/inscripciones.csv')
df_cal = pd.read_csv('../data/01_raw/calificaciones.csv')
df_asis = pd.read_csv('../data/01_raw/asistencia.csv')

df_cal['nota'] = pd.to_numeric(df_cal['nota'].astype(str).str.replace(',', '.'), errors='coerce')
notas_prom = df_cal.groupby('id_inscripcion')['nota'].mean().reset_index().rename(columns={'nota': 'nota_final'})

asis_tot = df_asis.groupby('id_inscripcion').size().reset_index(name='total')
ausentes = df_asis[df_asis['estado_asistencia'].str.lower().str.contains('ausente|injustificado', na=False)]
asis_aus = ausentes.groupby('id_inscripcion').size().reset_index(name='faltas')
asis_res = pd.merge(asis_tot, asis_aus, on='id_inscripcion', how='left').fillna(0)
asis_res['porcentaje_asistencia'] = 100 - ((asis_res['faltas'] / asis_res['total']) * 100)

df_master = pd.merge(df_insc, df_est, on='id_estudiante', how='left')
df_master = pd.merge(df_master, notas_prom, on='id_inscripcion', how='left')
df_master = pd.merge(df_master, asis_res[['id_inscripcion', 'porcentaje_asistencia']], on='id_inscripcion', how='left')

# Borrar filas con datos nulos en nuestras variables clave
df_master = df_master.dropna(subset=['porcentaje_asistencia', 'nota_final'])

# Variable Objetivo: 1 (Aprueba), 0 (Reprueba)
df_master['aprueba'] = np.where(df_master['nota_final'] >= 4.0, 1, 0)

# 3. MACHINE LEARNING: Preparar Features (X) y Target (y)
# Usaremos la asistencia para predecir si aprueba
X = df_master[['porcentaje_asistencia']] 
y = df_master['aprueba']

# Separar en datos de Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Crear el PIPELINE exigido por la rúbrica
# El pipeline primero estandariza los datos y luego aplica el modelo RandomForest
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clasificador', RandomForestClassifier(random_state=42, n_estimators=100))
])

# 5. Entrenar el modelo
pipeline_rf.fit(X_train, y_train)

# 6. Hacer predicciones con los datos de prueba
y_pred = pipeline_rf.predict(X_test)

# 7. Evaluar el modelo
print("✅ ¡Modelo entrenado con éxito!\n")
print("--- MÉTRICAS DE EVALUACIÓN ---")
print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred):.2f}\n")
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

✅ ¡Modelo entrenado con éxito!

--- MÉTRICAS DE EVALUACIÓN ---
Exactitud (Accuracy): 0.55

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      0.09      0.16        34
           1       0.53      1.00      0.69        35

    accuracy                           0.55        69
   macro avg       0.77      0.54      0.43        69
weighted avg       0.76      0.55      0.43        69



In [2]:
# --- 8. OPTIMIZACIÓN DE HIPERPARÁMETROS CON GridSearchCV ---
from sklearn.model_selection import GridSearchCV

# 1. Definir el "diccionario" de parámetros que queremos probar
# Vamos a probar distintos tamaños de bosque y profundidades de los árboles
parametros = {
    'clasificador__n_estimators': [50, 100, 200],       # Cantidad de árboles
    'clasificador__max_depth': [3, 5, 10, None],        # Profundidad máxima para no sobreajustar
    'clasificador__class_weight': ['balanced', None]    # 'balanced' fuerza al modelo a prestarle atención a los reprobados
}

# 2. Configurar la Búsqueda en Cuadrícula (GridSearchCV) con Validación Cruzada (cv=5)
grid_search = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=parametros,
    cv=5,               # Validación cruzada de 5 particiones (K-Fold) exigida por la pauta
    scoring='f1_macro', # Buscamos optimizar el equilibrio entre clases, no solo la exactitud
    n_jobs=-1           # Usa todos los núcleos de tu computador para ir más rápido
)

print("⏳ Entrenando y buscando los mejores parámetros (esto puede tomar unos segundos)...")

# 3. Entrenar el GridSearch
grid_search.fit(X_train, y_train)

# 4. Extraer el mejor modelo encontrado
mejor_pipeline = grid_search.best_estimator_

print("\n✅ ¡Búsqueda completada!")
print("Los mejores hiperparámetros encontrados son:")
print(grid_search.best_params_)

# 5. Predecir y evaluar con el MEJOR modelo
y_pred_opt = mejor_pipeline.predict(X_test)

print("\n--- NUEVAS MÉTRICAS DEL MODELO OPTIMIZADO ---")
print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred_opt):.2f}\n")
print("Reporte de Clasificación Optimizado:")
print(classification_report(y_test, y_pred_opt))

⏳ Entrenando y buscando los mejores parámetros (esto puede tomar unos segundos)...

✅ ¡Búsqueda completada!
Los mejores hiperparámetros encontrados son:
{'clasificador__class_weight': 'balanced', 'clasificador__max_depth': 3, 'clasificador__n_estimators': 100}

--- NUEVAS MÉTRICAS DEL MODELO OPTIMIZADO ---
Exactitud (Accuracy): 0.55

Reporte de Clasificación Optimizado:
              precision    recall  f1-score   support

           0       1.00      0.09      0.16        34
           1       0.53      1.00      0.69        35

    accuracy                           0.55        69
   macro avg       0.77      0.54      0.43        69
weighted avg       0.76      0.55      0.43        69



In [ ]:
# --- 9. MODELO V2: AGREGANDO MÁS VARIABLES (FEATURE ENGINEERING) ---
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Seleccionamos más variables para nuestro modelo
# Usaremos Asistencia, Carrera y Sede
df_features = df_master[['porcentaje_asistencia', 'carrera', 'sede', 'aprueba']].dropna()

# 2. Convertir texto a números (One-Hot Encoding con get_dummies)
# Los modelos matemáticos no entienden palabras como "Derecho" o "Santiago Centro"
X_multi = pd.get_dummies(df_features[['porcentaje_asistencia', 'carrera', 'sede']], drop_first=True)
y_multi = df_features['aprueba']

# 3. Separar datos
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi)

# 4. Crear el Pipeline Optimizado (usando los hiperparámetros que descubrimos)
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('clasificador', RandomForestClassifier(n_estimators=100, max_depth=3, class_weight='balanced', random_state=42))
])

# 5. Entrenar y Predecir
pipeline_final.fit(X_train_m, y_train_m)
y_pred_m = pipeline_final.predict(X_test_m)

# 6. Evaluar
print("✅ ¡Modelo V2 Entrenado con Múltiples Variables!\n")
print(f"Exactitud (Accuracy): {accuracy_score(y_test_m, y_pred_m):.2f}\n")
print("Reporte de Clasificación Final:")
print(classification_report(y_test_m, y_pred_m))

# 7. EXPORTAR EL MODELO (Para cumplir con la reproducibilidad)
# ESTA ES LA SOLUCIÓN AL ERROR: Crea la carpeta 'models' si no existe
os.makedirs('../models', exist_ok=True) 

joblib.dump(pipeline_final, '../models/modelo_clasificacion_rf.pkl')
print("\n💾 Modelo guardado exitosamente en la carpeta 'models/'")

✅ ¡Modelo V2 Entrenado con Múltiples Variables!

Exactitud (Accuracy): 0.53

Reporte de Clasificación Final:
              precision    recall  f1-score   support

           0       0.52      0.39      0.45        28
           1       0.53      0.66      0.58        29

    accuracy                           0.53        57
   macro avg       0.53      0.52      0.52        57
weighted avg       0.53      0.53      0.52        57


💾 Modelo guardado exitosamente en la carpeta 'models/'
